In [4]:
import cv2
import re
import string
import spacy
import pytesseract

import numpy as np
import pandas as pd

from glob import glob
from string import whitespace

In [5]:


def clean_text(txt):
    white_space = whitespace
    punch = '!"#$%&\'()*+,:;<=>?[\\]^_`{|}~'
    
    table_white_space = str.maketrans('','', white_space)
    table_puncuation = str.maketrans('','',punch)

    text = str(txt)
    text = text.lower()

    remove_whitespace = text.translate(table_white_space)
    remove_punctation = remove_whitespace.translate(table_puncuation)

    return str(remove_punctation)

#### Need for step to complete 

1. Load Image
2. Extract data using Pytessract
3. convert dat into content
4. get predicition for NER model


In [6]:
### Load NER model
model_ner = spacy.load('./output/model-best/')
model_ner

In [7]:
## Load Image
# image = cv2.imread('./BusinessCardData/Selected/006.jpeg')
image = cv2.imread('./data/Pradeep.jpeg')

# cv2.imshow("Businesscard", image)
# cv2.waitKey(0)
# cv2.destroyAllWindows()

In [8]:
# Extract data using Pytesseract
test_data = pytesseract.image_to_data(image)
test_data

'level\tpage_num\tblock_num\tpar_num\tline_num\tword_num\tleft\ttop\twidth\theight\tconf\ttext\n1\t1\t0\t0\t0\t0\t0\t0\t1024\t594\t-1\t\n2\t1\t1\t0\t0\t0\t524\t41\t407\t71\t-1\t\n3\t1\t1\t1\t0\t0\t39\t41\t892\t93\t-1\t\n4\t1\t1\t1\t1\t0\t654\t41\t277\t29\t-1\t\n5\t1\t1\t1\t1\t1\t654\t42\t80\t27\t88.081161\tMob.:\n5\t1\t1\t1\t1\t2\t747\t41\t184\t29\t96.767883\t9289548853\n4\t1\t1\t1\t2\t0\t524\t81\t404\t31\t-1\t\n5\t1\t1\t1\t2\t1\t524\t82\t81\t25\t93.044182\tE-mail\n5\t1\t1\t1\t2\t2\t616\t89\t5\t17\t91.642731\t:\n5\t1\t1\t1\t2\t3\t632\t81\t296\t31\t84.705475\tpkvksk2010@yahoo.in\n2\t1\t2\t0\t0\t0\t292\t237\t383\t43\t-1\t\n3\t1\t2\t1\t0\t0\t292\t237\t383\t43\t-1\t\n4\t1\t2\t1\t1\t0\t292\t237\t383\t43\t-1\t\n5\t1\t2\t1\t1\t1\t292\t237\t205\t43\t96.698212\tPRADEEP\n5\t1\t2\t1\t1\t2\t512\t245\t163\t33\t96.270096\tKUMAR\n2\t1\t3\t0\t0\t0\t348\t292\t267\t31\t-1\t\n3\t1\t3\t1\t0\t0\t348\t292\t267\t31\t-1\t\n4\t1\t3\t1\t1\t0\t348\t292\t267\t31\t-1\t\n5\t1\t3\t1\t1\t1\t348\t292\t142\t31\t96.2619

In [9]:
# Convert into data frame
test_list = list(map(lambda x:x.split('\t'), test_data.split('\n')))

df = pd.DataFrame(test_list[1:], columns=test_list[0])
df

,level,page_num,block_num,par_num,line_num,word_num,left,top,width,height,conf,text
0,1,1,0,0,0,0,0,0,1024,594,-1,
1,2,1,1,0,0,0,524,41,407,71,-1,
2,3,1,1,1,0,0,39,41,892,93,-1,
3,4,1,1,1,1,0,654,41,277,29,-1,
4,5,1,1,1,1,1,654,42,80,27,88.081161,Mob.:
5,5,1,1,1,1,2,747,41,184,29,96.767883,9289548853
6,4,1,1,1,2,0,524,81,404,31,-1,
7,5,1,1,1,2,1,524,82,81,25,93.044182,E-mail
8,5,1,1,1,2,2,616,89,5,17,91.642731,:
9,5,1,1,1,2,3,632,81,296,31,84.705475,pkvksk2010@yahoo.in


In [10]:
df = df.dropna()

In [11]:
df['text'] = df['text'].apply(clean_text)
df['text']

C:\Users\Bindra\AppData\Local\Temp\ipykernel_3524\64179834.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['text'] = df['text'].apply(clean_text)


0                        
1                        
2                        
3                        
4                    mob.
5              9289548853
6                        
7                  e-mail
8                        
9     pkvksk2010@yahoo.in
10                       
11                       
12                       
13                pradeep
14                  kumar
15                       
16                       
17                       
18              insurance
19                advisor
20                       
21                       
22                       
23                   life
24              insurance
25            corporation
26                     of
27                  india
28                       
29                       
30                       
31                 branch
32                   unit
33                       
34                    11a
35                 a-3/24
36                    ist
37                  floor
38          

In [12]:
# Covert data into content
df_clean = df.query("text != '' ")

content = " ".join([w for w in df_clean['text']])
print(content)

mob. 9289548853 e-mail pkvksk2010@yahoo.in pradeep kumar insurance advisor life insurance corporation of india branch unit 11a a-3/24 ist floor janak puri new delhi-110058


In [13]:
# Get prediction from NER model
doc = model_ner(content)


In [14]:
# spacy displacy
from spacy import displacy



In [15]:
displacy.serve(doc, style = 'ent')


C:\Users\Bindra\Desktop\TODO\python_data_science\Spacy_entity\entity\Lib\site-packages\spacy\displacy\__init__.py:106: UserWarning: [W011] It looks like you're calling displacy.serve from within a Jupyter notebook or a similar environment. This likely means you're already running a local web server, so there's no need to make displaCy start another one. Instead, you should be able to replace displacy.serve with displacy.render to show the visualization.
  warnings.warn(Warnings.W011)



Using the 'ent' visualizer
Serving on http://0.0.0.0:5000 ...

Shutting down server on port 5000.


In [16]:
displacy.render(doc, style = 'ent')

#### Tagging

In [17]:
doc.to_json()

{'text': 'mob. 9289548853 e-mail pkvksk2010@yahoo.in pradeep kumar insurance advisor life insurance corporation of india branch unit 11a a-3/24 ist floor janak puri new delhi-110058',
 'ents': [{'start': 5, 'end': 15, 'label': 'B-PHONE'},
  {'start': 23, 'end': 42, 'label': 'B-EMAIL'},
  {'start': 43, 'end': 50, 'label': 'B-NAME'},
  {'start': 51, 'end': 56, 'label': 'I-NAME'},
  {'start': 57, 'end': 66, 'label': 'B-DES'},
  {'start': 67, 'end': 74, 'label': 'I-DES'},
  {'start': 75, 'end': 79, 'label': 'B-ORG'},
  {'start': 80, 'end': 89, 'label': 'I-ORG'},
  {'start': 90, 'end': 101, 'label': 'I-ORG'},
  {'start': 102, 'end': 104, 'label': 'I-ORG'},
  {'start': 105, 'end': 110, 'label': 'I-ORG'},
  {'start': 144, 'end': 149, 'label': 'B-NAME'}],
 'tokens': [{'id': 0, 'start': 0, 'end': 3},
  {'id': 1, 'start': 3, 'end': 4},
  {'id': 2, 'start': 5, 'end': 15},
  {'id': 3, 'start': 16, 'end': 17},
  {'id': 4, 'start': 17, 'end': 18},
  {'id': 5, 'start': 18, 'end': 22},
  {'id': 6, 'st

In [20]:
doc_json = doc.to_json()
doc_json.keys()

dict_keys(['text', 'ents', 'tokens'])

In [21]:
doc_json['text']

'mob. 9289548853 e-mail pkvksk2010@yahoo.in pradeep kumar insurance advisor life insurance corporation of india branch unit 11a a-3/24 ist floor janak puri new delhi-110058'

In [22]:
doc_json['ents']

[{'start': 5, 'end': 15, 'label': 'B-PHONE'},
 {'start': 23, 'end': 42, 'label': 'B-EMAIL'},
 {'start': 43, 'end': 50, 'label': 'B-NAME'},
 {'start': 51, 'end': 56, 'label': 'I-NAME'},
 {'start': 57, 'end': 66, 'label': 'B-DES'},
 {'start': 67, 'end': 74, 'label': 'I-DES'},
 {'start': 75, 'end': 79, 'label': 'B-ORG'},
 {'start': 80, 'end': 89, 'label': 'I-ORG'},
 {'start': 90, 'end': 101, 'label': 'I-ORG'},
 {'start': 102, 'end': 104, 'label': 'I-ORG'},
 {'start': 105, 'end': 110, 'label': 'I-ORG'},
 {'start': 144, 'end': 149, 'label': 'B-NAME'}]

In [23]:
doc_json['tokens']

[{'id': 0, 'start': 0, 'end': 3},
 {'id': 1, 'start': 3, 'end': 4},
 {'id': 2, 'start': 5, 'end': 15},
 {'id': 3, 'start': 16, 'end': 17},
 {'id': 4, 'start': 17, 'end': 18},
 {'id': 5, 'start': 18, 'end': 22},
 {'id': 6, 'start': 23, 'end': 42},
 {'id': 7, 'start': 43, 'end': 50},
 {'id': 8, 'start': 51, 'end': 56},
 {'id': 9, 'start': 57, 'end': 66},
 {'id': 10, 'start': 67, 'end': 74},
 {'id': 11, 'start': 75, 'end': 79},
 {'id': 12, 'start': 80, 'end': 89},
 {'id': 13, 'start': 90, 'end': 101},
 {'id': 14, 'start': 102, 'end': 104},
 {'id': 15, 'start': 105, 'end': 110},
 {'id': 16, 'start': 111, 'end': 117},
 {'id': 17, 'start': 118, 'end': 122},
 {'id': 18, 'start': 123, 'end': 126},
 {'id': 19, 'start': 127, 'end': 133},
 {'id': 20, 'start': 134, 'end': 137},
 {'id': 21, 'start': 138, 'end': 143},
 {'id': 22, 'start': 144, 'end': 149},
 {'id': 23, 'start': 150, 'end': 154},
 {'id': 24, 'start': 155, 'end': 158},
 {'id': 25, 'start': 159, 'end': 171}]

In [24]:
doc_text = doc_json['text']

In [26]:
df_token = pd.DataFrame(doc_json['tokens'])
df_token.head()

,id,start,end
0,0,0,3
1,1,3,4
2,2,5,15
3,3,16,17
4,4,17,18
